# AI ANALYST LAB

![](../_img/ai_analyst_lab.png)

### A Hands-on Course on AI for Data Analysts
## Session 06: Customer segmentation with k-means clustering

Feedback should be sent to [goran.milovanovic@datakolektiv.com](mailto:goran.milovanovic@datakolektiv.com).

This notebook accompanies the **AI Analyst LAB** course. Welcome to Session 06 — the week we stop predicting a known answer and start **discovering structure that has no answer key at all**.


***
### What we will do today

Take a breath and look back at how far you have come. In **Session 01** you learned descriptive statistics, three probability distributions (the **Normal**, the **Binomial**, the **Poisson**), and the sampling distribution of the mean. In **Session 02** you added conditional probability, expected value, and the bootstrap. In **Session 03** you built the hypothesis-testing framework. In **Session 04** you crossed into *relationships between numbers* — covariance, correlation, and **linear regression**. And last week, in **Session 05**, you fit a **logistic regression** to predict a yes/no outcome (churn), read its coefficients, and turned probabilities into a business decision.

Every one of those models had something in common: **a known answer to learn from.** Session 04 had a `quality` column; Session 05 had a `Churn` column. We showed the model the right answers and trained it to reproduce them. That whole family of methods is called **supervised learning** — there is a *supervisor*, a target column, grading every guess.

**This week we take the supervisor away.**

Our dataset is a list of customers and how much each one spends, but there is **no "right group" column** telling us which customers belong together. Our job is to *find* the groups ourselves — to look at the spending patterns and discover the natural **segments** hiding in the data. This is **unsupervised learning**, and the specific tool is **k-means clustering**.

The deliverable is different too. In Session 05 we produced a probability score. This week we produce a **segmentation story** — a small number of named customer types, each with a plain-English description and a recommended sales and retention action. That is something a commercial team can act on Monday morning.

We are going to keep our usual promise: every new idea is introduced from scratch, with narrative first and a tiny concrete example before any formula. When we reach genuinely new words — *centroid*, *inertia*, *silhouette* — we will walk back to something you already know (the **standard deviation** and **z-score** from Session 01, the **sum of squared residuals** from Session 04) and build the new idea on top of the old one.

The business setting is a **wholesale food-and-goods distributor**. The sections, in order:

| Section | What happens |
|---|---|
| 6.1 | The business case — segment the customer base to tailor sales and retention |
| 6.2 | Meet your Session 06 Tutor (Claude Project) |
| 6.3 | Setup — imports and loading the wholesale data |
| 6.4 | A first look at the six product categories (and a warning sign in the histograms) |
| 6.5 | A new kind of question — **supervised vs unsupervised** learning |
| 6.6 | **Distance is the whole idea** — what k-means actually measures |
| 6.7 | **Why we must scale first** — the z-score returns from Session 01 |
| 6.8 | **How k-means works** — the assign-and-update loop |
| 6.9 | **What k-means is minimizing** — within-cluster sum of squares (the new $SS_{res}$) |
| 6.10 | **Choosing k, part 1** — the elbow method |
| 6.11 | **Choosing k, part 2** — the silhouette score |
| 6.12 | Fitting the final model |
| 6.13 | **Profiling the segments** — what each cluster actually buys |
| 6.14 | **Visualizing the segments** — the fingerprint heatmap |
| 6.15 | **Naming the segments** and an external-validation surprise |
| 6.16 | Our API calls — Anthropic turns the profiles into a sales playbook |
| 6.17 | The segmentation story and sales playbook — fully worked |
| 6.18 | References — what to study to deepen this session |

A few rules for using this notebook, same as the previous five weeks:

- **Run the cells in order.** Each section builds on the one before.
- **Read the explanations, do not just run the cells.** The intuition is the point.
- **Every line of code has a comment above it** in beginner language.
- **Use your Session 06 Tutor** (the Claude Project at `_tutors/session06_tutor.xml`) when something feels confusing.
- **Compute first in Python, then use the model to interpret.** Same rule as Sessions 01–05 — the model never invents numbers.


***
## 6.1 The business case

You are the analyst for a **wholesale distributor**. Your company buys food and household goods in bulk and sells them on to other businesses: corner grocery shops, supermarkets, cafés, restaurants, hotels, and caterers. You have just been handed a year of records: for each of your **440 client businesses**, how much they spent (in monetary units) across **six product categories** — fresh produce, milk, grocery, frozen, detergents-and-paper, and delicatessen.

The Head of Commercial drops by your desk with a problem:

> *"We treat every client the same — same catalogue, same sales calls, same promotions. But they are obviously not the same. A busy restaurant buys fresh and frozen by the pallet and barely touches detergents. A neighbourhood grocery is the opposite. I want to **stop selling one-size-fits-all**. Can you group our clients into a handful of meaningful types, so each of my sales reps knows what to pitch and what to protect? Give me the segments, tell me who is in each, and tell me what we should do differently for each one."*

Notice what is **not** in that brief: there is no list of "correct" segments to aim for. Nobody has labelled client #207 as a "restaurant-type" buyer. The grouping does not exist yet — **you have to create it from the spending patterns alone.** That is the defining feature of this week's problem, and it is exactly what clustering is for.

Your final deliverables, by Friday:

1. **A segmentation** — a small number of customer segments, each defined by how it spends.
2. **A profile per segment** — who is in it, and what makes it distinct.
3. **A sales playbook** — one concrete sales action and one retention action per segment.

Everything else in this notebook serves those three artifacts.


***
## 6.2 Meet your Session 06 Tutor

As in every session, you have a **Claude Project** acting as your personal statistics tutor for this week's ideas. The prompt lives in the repository at `_tutors/session06_tutor.xml`, and the setup walkthrough is in `_tutors/TutorProjectCreation.md`.

Your **Session 06 Tutor** is scoped to *this* session only — clustering, distance, scaling, choosing the number of clusters, and reading cluster profiles. It will not race ahead to topics we have not covered. Use it whenever a sentence here does not click: paste the sentence in and ask *"explain this more slowly, with a tiny example."*

Two reminders about the wider tutor system:

- For questions about **Python, pandas, NumPy, matplotlib, or seaborn** — *"why does `groupby` work like that?"* — use the cross-session **Python Stack Tutor** (`_tutors/python_stack_tutor.xml`) instead. It is shared across all eight sessions.
- For questions about **earlier statistics** — the z-score from Session 01, regression from Session 04 — those sessions' own tutors are the right place.

The discipline we have kept since Session 01 holds again this week: **Python computes the numbers; the model only helps you interpret and communicate them.** The model never invents a number.


***
## 6.3 Setup — imports and loading the wholesale data

We start, as always, by loading the tools. Most of these are old friends from previous sessions. The new arrivals this week all come from **scikit-learn** (imported as `sklearn`), the standard Python library for machine learning:

- **`StandardScaler`** — puts every column on the same scale (the z-score machine; §6.7).
- **`KMeans`** — the clustering algorithm itself (§6.8).
- **`silhouette_score`** — one of the two tools we use to choose how many clusters to make (§6.11).

Run the cell below. It imports everything and prints the version of each library, so that if a number here ever differs from a number on your screen, you can check whether a version mismatch is the reason.


In [ ]:
# pandas gives us the DataFrame — the table-with-labels we have used since Session 01.
import pandas as pd

# numpy gives us fast array maths (square roots, logs, sums over columns).
import numpy as np

# matplotlib is our base plotting library.
import matplotlib.pyplot as plt

# seaborn sits on top of matplotlib and makes good-looking statistical plots with less code.
import seaborn as sns

# StandardScaler rescales each column to mean 0 and standard deviation 1 (the z-score, from Session 01).
from sklearn.preprocessing import StandardScaler

# KMeans is the clustering algorithm at the heart of this session.
from sklearn.cluster import KMeans

# silhouette_score measures how well-separated a clustering is — one of our "how many clusters?" tools.
from sklearn.metrics import silhouette_score

# Tell Jupyter to draw plots directly under each cell.
%matplotlib inline

# Apply a clean, consistent visual theme to every plot in the notebook.
sns.set_theme(style="whitegrid")

# Import the top-level sklearn package just so we can print its version below.
import sklearn

# Print the version of each library, so any number mismatch can be traced to a version difference.
print("pandas:     ", pd.__version__)
print("numpy:      ", np.__version__)
print("matplotlib: ", plt.matplotlib.__version__)
print("seaborn:    ", sns.__version__)
print("scikit-learn:", sklearn.__version__)

Now we load the dataset. It lives one folder up from this notebook, inside `_data/wholesale_customers/`. The file is a plain CSV with one row per client business.

The eight columns are:

- **`Channel`** — the *kind* of business: `1` = **Horeca** (Hotel / Restaurant / Café) or `2` = **Retail** (shops selling to the public).
- **`Region`** — where the client is: `1` = **Lisbon**, `2` = **Oporto**, `3` = **Other** (the rest of Portugal).
- **Six spending columns** — annual spend (in monetary units, abbreviated *m.u.*) on **`Fresh`**, **`Milk`**, **`Grocery`**, **`Frozen`**, **`Detergents_Paper`**, and **`Delicassen`** (delicatessen).

A crucial decision right away: **we will cluster using only the six spending columns.** We deliberately set `Channel` and `Region` aside and do **not** let the algorithm see them. Why? Because they are *labels we already have* — and at the very end (§6.15) we will use them for a beautiful test: *did our segmentation, built from spending alone, manage to rediscover the Horeca-vs-Retail split on its own?* Holding them back now makes that test honest.


In [ ]:
# Build the path to the dataset: one folder up (..), then into _data/wholesale_customers/.
data_path = "../_data/wholesale_customers/Wholesale customers data.csv"

# Read the CSV into a DataFrame called `wholesale`.
wholesale = pd.read_csv(data_path)

# Print the shape: (number of rows = clients, number of columns).
print("Shape (clients, columns):", wholesale.shape)

# Show the first eight rows so we can see the structure with our own eyes.
wholesale.head(8)

***
## 6.4 A first look at the six product categories

Before any clustering, we do what every session has trained you to do first: **look at the data**. We want to know the typical spend in each category, how spread out it is, and — this will matter enormously in a moment — what *shape* each category's distribution has.

Let us start with the summary table you have used since Session 01: `describe()`. It reports, for each column, the count, mean, standard deviation, minimum, the three quartiles, and the maximum.


In [ ]:
# Define the six spending columns we will work with throughout the session.
spend_cols = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Confirm there are no missing values to worry about (a clean dataset makes our life easier).
print("Total missing values across all columns:", wholesale.isnull().sum().sum())

# Show the descriptive statistics for the six spending columns, rounded to whole monetary units.
wholesale[spend_cols].describe().round(0)

Read that table slowly, because it is already telling us a story.

- **`Fresh`** has a mean of about **12,000** m.u. but a maximum of **112,151** — almost ten times the average. **`Milk`** averages about **5,800** but reaches **73,498**. Every category has a maximum that towers over its mean.
- The **standard deviations** are huge — for `Fresh` the standard deviation (about **12,647**) is *larger than the mean itself*. Recall from Session 01 (§1.5) that the standard deviation measures typical spread around the mean; when it is bigger than the mean, the data is extremely stretched out.

This is the signature of a **right-skewed** distribution: most customers cluster at modest spend, while a handful of very large customers stretch far out to the right. Let us see it directly with a histogram for each category — exactly the *"analyst's starter toolkit"* plot you built in Session 01.


In [ ]:
# Create a 2-row by 3-column grid of plots, one panel per spending category.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Flatten the 2D grid into a flat list so we can pair each axis with one category in a loop.
axes = axes.flatten()

# Draw one histogram per category.
for ax, col in zip(axes, spend_cols):
    # Plot the distribution of annual spend for this category into its own panel.
    sns.histplot(wholesale[col], bins=40, ax=ax, color="steelblue")
    # Title the panel with the category name.
    ax.set_title(col)
    # Label the horizontal axis with the unit.
    ax.set_xlabel("annual spend (m.u.)")

# Adjust spacing so titles and labels do not overlap.
plt.tight_layout()

# Render the figure.
plt.show()

Every single panel has the same shape: a tall pile of customers near zero and a long, thin tail reaching far to the right. We can put a number on that shape using **skewness** — a statistic that is `0` for a perfectly symmetric distribution and grows positive as the right tail gets longer.


In [ ]:
# Compute the skewness of each spending category; larger positive = longer right tail.
skew_vals = wholesale[spend_cols].skew().sort_values(ascending=False)

# Print the skewness values, most-skewed category first.
print("Skewness of each category (0 = symmetric, large positive = long right tail):")
print(skew_vals.round(2))

The numbers confirm the eye. **`Delicassen`** is wildly skewed (about **11.2**), **`Frozen`** about **5.9**, and even the *least* skewed category, **`Fresh`**, sits at about **2.6** — still strongly right-tailed. For comparison, a Normal distribution (Session 01, §1.7) has a skewness of `0`.

**Why flag this now?** Because k-means measures **distance** between customers, and distance is brutally sensitive to a few giant numbers. A handful of enormous customers in a long tail can hijack the whole clustering — the algorithm ends up obsessing over the outliers instead of finding the broad patterns we actually want. We will see this happen for real in §6.7, and we will fix it. For now, just hold onto the warning: **the data is heavily skewed, and that will matter.**


***
## 6.5 A new kind of question — supervised vs unsupervised learning

Let us name precisely what makes this week different, because the distinction is one of the most important in all of machine learning.

**Last week (Session 05), we did supervised learning.** We had a column called `Churn` that told us, for every customer, the *right answer*: did they leave (`1`) or stay (`0`)? We handed the model those answers and trained it to reproduce them from the other columns. The word *supervised* is literal — there was a supervisor, the `Churn` column, grading every prediction the model made. Session 04's regression was supervised too: the `quality` column was the supervisor.

**This week, there is no supervisor.** Our `wholesale` table has six spending columns and *no column that says which segment a customer belongs to.* The segments do not exist yet. So we cannot train a model to reproduce an answer — there is no answer to reproduce. Instead we ask a different question:

> *"Given only how these customers behave, which ones are similar to each other? Can we sort them into a few natural groups?"*

That is **unsupervised learning**: finding structure in data that has no labels. **Clustering** is the most common kind — sorting items into groups (*clusters*) so that members of the same group are similar and members of different groups are different.

Here is a tiny, everyday example to make it concrete. Imagine emptying a jar of mixed coins onto a table with no instructions. You would *naturally* start making piles — the big ones here, the small bronze ones there, the silver ones in a third pile — without anyone telling you "this is a 1-euro coin." You are clustering: grouping by similarity, inventing the groups as you go. K-means does exactly this, but with customers instead of coins, and using spending instead of size and colour.

Two consequences flow from having no supervisor, and they shape the rest of the notebook:

1. **There is no accuracy to compute.** In Session 05 we could check predictions against the true `Churn` labels and build a confusion matrix. Here there are no true labels, so "accuracy" is meaningless. We will need *new* ways to judge whether a clustering is any good — that is what §6.10 and §6.11 are for.
2. **The number of groups is our choice.** Nobody tells us there are three segments, or five. We have to *decide* how many clusters to look for, and justify it. That decision — choosing **k** — is the central craft of this session.

One honest note, the same one we made in Session 05 and which is course policy: everything we build describes **these 440 customers**. We are not forecasting which segment a brand-new, unseen customer would fall into; we are organising the customers we actually have, so the business can act on them today.


***
## 6.6 Distance is the whole idea — what k-means measures

To group customers by *similarity*, we need to make "similar" precise. K-means uses the most intuitive definition there is: **two customers are similar if they are close together**, where "close" means a short **distance** between them.

Picture each customer as a **point**. In Session 04 we plotted customers as points on a 2D scatterplot — one axis per variable. A wholesale customer here has *six* spending numbers, so each customer is a point in a **six-dimensional space** — one axis for `Fresh`, one for `Milk`, and so on. We cannot draw six dimensions, but the maths of distance works in any number of dimensions exactly as it does on a flat map.

On a flat map, the straight-line distance between two points comes from the Pythagorean theorem: go across, go up, and the diagonal is $\sqrt{(\text{across})^2 + (\text{up})^2}$. The same formula, extended to more dimensions, is called the **Euclidean distance**:

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{j=1}^{p} (x_j - y_j)^2}$$

Let us name every symbol, slowly:

- $\mathbf{x}$ and $\mathbf{y}$ (bold) are **two customers** — each one a list of six spending numbers.
- $d(\mathbf{x}, \mathbf{y})$ is the **distance** between them — one single number. Small means similar; large means different.
- $p$ is the **number of dimensions** — here $p = 6$, the six categories.
- $j$ is a **counter** that walks through the categories one at a time: $j = 1$ is `Fresh`, $j = 2$ is `Milk`, and so on up to $j = 6$.
- $x_j$ is customer $\mathbf{x}$'s spend in category $j$; $y_j$ is customer $\mathbf{y}$'s spend in the same category.
- $x_j - y_j$ is **how far apart the two customers are in that one category.** We square it (so positive and negative gaps both count as distance, and big gaps count much more), then $\sum_{j=1}^{p}$ adds those squared gaps across all six categories, and the square root $\sqrt{\;}$ converts the total back to the original units.

That is the entire idea. Let us compute one real distance by hand to feel it — between customer **0** and customer **3** in our data.


In [ ]:
# Pull out customer 0 and customer 3, keeping only the six spending columns.
cust_a = wholesale.loc[0, spend_cols]
cust_b = wholesale.loc[3, spend_cols]

# Compute the squared difference in each category: (x_j - y_j)^2.
sq_diff = (cust_a - cust_b) ** 2

# The Euclidean distance is the square root of the sum of those squared differences.
dist = np.sqrt(sq_diff.sum())

# Work out what fraction of the total squared distance each category contributes.
contribution = (sq_diff / sq_diff.sum() * 100).round(1)

# Print the overall distance.
print(f"Raw Euclidean distance between customer 0 and customer 3: {dist:,.1f}")

# Print how much each category contributed to that distance, biggest first.
print("\nEach category's share of the squared distance (%):")
print(contribution.sort_values(ascending=False))

The distance comes out around **11,238**. But look at the breakdown — and this is the punchline that sets up the next section. Almost the *entire* distance is driven by two categories: **`Milk`** (about **57%** of it) and **`Frozen`** (about **30%**). Meanwhile **`Detergents_Paper`** contributes under **4%** and **`Delicassen`** almost nothing (about **0.2%**).

Is that because milk and frozen goods are *genuinely* the most important way these two customers differ? **No.** It is purely because those categories happen to carry the **biggest raw numbers**, and squaring big gaps makes them dominate the sum. The distance is being decided by the *units on the axes*, not by what actually distinguishes customers. That is a problem — and fixing it is the single most important step in the whole pipeline.


***
## 6.7 Why we must scale first — the z-score returns from Session 01

We just saw distance get hijacked by whichever categories carry the biggest numbers. Let us measure how unequal the categories are by comparing their standard deviations.


In [ ]:
# Show the standard deviation of each category on the raw scale.
print("Standard deviation of each category (raw m.u.):")
print(wholesale[spend_cols].std().round(0).astype(int))

The spreads are wildly different: **`Fresh`** has a standard deviation around **12,647**, while **`Delicassen`** is around **2,820** — more than a fourfold difference. A category with a big standard deviation will dominate every distance calculation simply by being measured in bigger numbers, drowning out categories that might matter just as much to the business.

The fix is the one you already met in **Session 01, §1.5 and §1.10**: the **z-score** (standardization). For each value we subtract that column's mean and divide by that column's standard deviation:

$$z_j = \frac{x_j - \bar{x}_j}{s_j}$$

Symbol by symbol:

- $x_j$ is a raw spend value in category $j$.
- $\bar{x}_j$ ("x-bar") is the **mean** of category $j$ across all customers.
- $s_j$ is the **standard deviation** of category $j$.
- $z_j$ is the **standardized value**: how many standard deviations above (positive) or below (negative) the category average this customer sits.

After standardizing, **every category has mean 0 and standard deviation 1.** None can dominate by sheer size any more — they all speak the same language: *standard deviations from average.* This is precisely the role the z-score played in Session 01 when we compared a value to a Normal distribution; here it lets six categories contribute *fairly* to a distance.

There is one more step we do *first*. Remember the heavy right-skew from §6.4? Those long tails put a few giant customers so far out that even after standardizing they would still distort the distances. The standard remedy is a **log transform**: replacing each spend value $x$ with $\log(1 + x)$. The logarithm pulls the giant values back toward the pack while leaving the ordering intact, turning each lopsided distribution into something much more symmetric. We use `log1p`, which computes $\log(1 + x)$ and is safe even when a value is `0`.

So our scaling pipeline has two steps, in order: **(1) log-transform to tame the skew, then (2) standardize to equalise the scales.** Let us build it.


In [ ]:
# Step 1 — tame the heavy right tails. log1p computes log(1 + x): safe at 0, and it pulls
# the giant customers back toward the rest while keeping every customer's ordering intact.
wholesale_log = np.log1p(wholesale[spend_cols])

# Step 2 — standardize each logged column to mean 0 and standard deviation 1 (the z-score).
# StandardScaler does the (x - mean) / sd arithmetic, column by column, for us.
scaler = StandardScaler()

# Fit the scaler to the logged data and transform it in one call; the output is a NumPy array.
X_scaled_array = scaler.fit_transform(wholesale_log)

# Wrap the array back into a DataFrame so we can read columns by their category names.
X_scaled = pd.DataFrame(X_scaled_array, columns=spend_cols)

# Confirm the transform worked: every column should now have mean about 0 ...
print("Mean of each scaled column (should be ~0):")
print(X_scaled.mean().round(2))

# ... and standard deviation about 1.
print("\nStandard deviation of each scaled column (should be ~1):")
print(X_scaled.std().round(2))

Every scaled column now has mean `0` and standard deviation `1` — the six categories finally contribute on equal terms. `X_scaled` is the table we will actually cluster.

To prove this matters and is not just bookkeeping, here is what happens if we **skip scaling** and cluster the raw numbers directly. (We will explain the `KMeans` call itself in the next section; for now just watch the cluster sizes.)


In [ ]:
# Cluster the RAW (unscaled) spending into 3 groups, deliberately doing the wrong thing.
kmeans_raw = KMeans(n_clusters=3, random_state=42, n_init=10)

# Get a cluster label for every customer from the raw-data clustering.
labels_raw = kmeans_raw.fit_predict(wholesale[spend_cols])

# Count how many customers landed in each raw-data cluster.
print("Cluster sizes when we (wrongly) cluster the RAW, unscaled spend:")
print(pd.Series(labels_raw).value_counts().sort_index())

Look at that result: the three clusters have sizes around **45, 393, and 2**. One "segment" contains just **two of the 440 customers** — the two most extreme spenders in the whole dataset. The algorithm did not find meaningful customer types; it spent its effort fencing off a couple of giant outliers, dumping nearly everyone else into one undifferentiated lump. That is **useless** as a segmentation.

This is the concrete payoff of the warning from §6.4. On the raw scale, distance is dominated by the few enormous customers; clustering chases them. After our log-and-standardize pipeline, as you will see in §6.12, the very same algorithm finds three balanced, interpretable, *actionable* segments. **Scaling is not optional housekeeping — it is the difference between a useless result and a useful one.**


***
## 6.8 How k-means works — the assign-and-update loop

Now the algorithm itself. The name gives away both halves: **k** is *how many clusters we want* (a number we choose), and **means** refers to the **average** customer at the centre of each cluster, called a **centroid**. A centroid is just an imaginary "typical member" of a cluster — its position is the average of all the customers currently in that cluster.

K-means finds clusters by repeating two simple steps until things stop changing. Here is the whole algorithm in plain words:

1. **Start.** Pick `k` starting centroids — for example, by dropping `k` points at random into the data.
2. **Assign step.** Give every customer to the **nearest** centroid (using the Euclidean distance from §6.6). This carves the customers into `k` groups.
3. **Update step.** Move each centroid to the **average position of the customers now assigned to it.** The centroid relocates to the true centre of its group.
4. **Repeat.** Go back to step 2 and reassign everyone to the nearest (now moved) centroid, then update again. Keep going until assignments stop changing.

That loop — *assign to nearest, then recompute the centre* — is the entire engine. It is a bit like seating guests at a party: you start with a few tables, everyone walks to the nearest table, then each table shuffles to the centre of its group, which makes a few guests realise a different table is now closer, so they move, and so on, until everyone is settled.

Two practical details that show up in our code:

- **The starting positions are random**, and a bad random start can occasionally settle into a poor arrangement. So scikit-learn runs the whole thing several times from different random starts and keeps the best result. That is what the argument **`n_init=10`** does — *try ten times, keep the best.*
- To make our run **reproducible** — the same answer every time you run it — we fix the random starting point with **`random_state=42`**. (The number `42` is arbitrary; any fixed value works. We have used this trick since Session 01's simulations.)

Before we can run it, we need to settle the one decision the algorithm cannot make for us: **what should `k` be?** To answer that well, we first need to know what k-means is actually trying to *achieve* — which is the next section.


***
## 6.9 What k-means is minimizing — within-cluster sum of squares

Every method in this course has had a quantity it tries to make as small as possible. In **Session 04**, linear regression chose the line that minimised the **sum of squared residuals** ($SS_{res}$) — the total of the squared gaps between each point and the line. K-means has its own version of exactly that idea.

K-means tries to make its clusters as **tight** as possible. It measures tightness with the **within-cluster sum of squares**, also called the **inertia**:

$$\text{WCSS} = \sum_{i=1}^{k} \; \sum_{\mathbf{x} \in C_i} \; \lVert \mathbf{x} - \boldsymbol{\mu}_i \rVert^2$$

Let us decode it piece by piece — it is friendlier than it looks:

- $k$ is the number of clusters.
- $C_i$ is **cluster number $i$** — the set of customers assigned to it. The inner sum $\sum_{\mathbf{x} \in C_i}$ means *"add this up over every customer $\mathbf{x}$ in cluster $i$."*
- $\boldsymbol{\mu}_i$ ("mu-eye") is the **centroid** of cluster $i$ — its average member from §6.8.
- $\lVert \mathbf{x} - \boldsymbol{\mu}_i \rVert^2$ is the **squared distance** from a customer to its own cluster's centroid — the squared Euclidean distance of §6.6.
- The outer sum $\sum_{i=1}^{k}$ then adds those totals across all $k$ clusters.

In words: **inertia is the total squared distance from every customer to the centre of its own cluster.** A small inertia means customers sit close to their centroids — tight, cohesive clusters. A large inertia means customers are scattered far from their centres — loose, fuzzy clusters. K-means' assign-and-update loop is, mathematically, a procedure for driving this inertia down as far as it can.

Notice the deep parallel with Session 04: there, $SS_{res}$ summed squared distances from points to a *line*; here, inertia sums squared distances from points to a *centroid*. Same primitive — *"measure fit as total squared distance from a summary"* — wearing slightly different clothes. You met it as regression's residual; you meet it again as clustering's inertia.

This gives us a tempting but **flawed** idea for choosing `k`: why not just pick the `k` that makes inertia smallest? The trouble is that inertia *always* shrinks as you add more clusters — with more centroids, everyone is closer to one of them. Taken to the extreme, if every customer were its own cluster, inertia would be zero and the "segmentation" would be worthless. So we cannot simply minimise inertia. We need a smarter reading of it — the **elbow method** — which is next.


***
## 6.10 Choosing k, part 1 — the elbow method

Since inertia always falls as `k` rises, we do not look for its minimum. Instead we look for the point of **diminishing returns** — the `k` after which adding another cluster barely helps.

The idea: compute the inertia for a range of `k` values, plot inertia against `k`, and look at the *shape* of the curve. Typically it drops steeply at first (going from 2 to 3 clusters genuinely tightens things a lot) and then flattens (going from 7 to 8 buys you almost nothing). The place where the steep drop bends into the flat part looks like an **elbow** in the curve — and that bend is a natural, defensible choice for `k`.

Let us compute inertia for every `k` from 2 to 8 and plot it.


In [ ]:
# The candidate values of k we will test: 2, 3, 4, 5, 6, 7, 8.
k_values = range(2, 9)

# An empty list to collect the inertia for each k.
inertias = []

# Fit k-means once for each candidate k and record its inertia.
for k in k_values:
    # Build a k-means model with k clusters, a fixed seed, and 10 random restarts.
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    # Fit it to the SCALED data (the table from §6.7).
    km.fit(X_scaled)
    # km.inertia_ is the within-cluster sum of squares for this fit — store it.
    inertias.append(km.inertia_)

# Start a new figure for the elbow plot.
plt.figure(figsize=(8, 5))

# Plot inertia against k, with a dot at each tested value.
plt.plot(list(k_values), inertias, marker="o")

# Label the axes and add a title.
plt.xlabel("number of clusters k")
plt.ylabel("inertia (within-cluster sum of squares)")
plt.title("Elbow plot: tightness improves quickly, then levels off")

# Render the plot.
plt.show()

# Also print the raw numbers so we can read the elbow precisely.
for k, val in zip(k_values, inertias):
    print(f"k = {k}: inertia = {val:,.1f}")

Read the numbers alongside the curve. Inertia falls from about **1,844** at `k = 2` to about **1,553** at `k = 3` — a drop of roughly **291**. The next step, `k = 3` to `k = 4`, saves only about **167**, and after that each extra cluster buys steadily less (about 121, then 91, then 88...). The curve bends most sharply right around **`k = 3`**: that is where steep improvement gives way to gentle, diminishing gains. The elbow points at **three clusters**.

The elbow is a judgement call, not a formula — reasonable analysts might argue for `k = 4`. So before committing, we get a **second opinion** from a completely different measure: the silhouette score.


***
## 6.11 Choosing k, part 2 — the silhouette score

The elbow method looks only at *tightness*. The **silhouette score** looks at something richer: for each customer, is it sitting comfortably in its own cluster, or is it almost as close to a *neighbouring* cluster? Good clusters are both **cohesive** (members close together) and **well-separated** (clusters far apart). The silhouette captures both at once.

For a single customer $i$, define:

- $a(i)$ = the average distance from customer $i$ to **the other members of its own cluster** (how snugly it fits in — small is good).
- $b(i)$ = the average distance from customer $i$ to the members of the **nearest *other* cluster** (how far it is from the next-best home — large is good).

The silhouette of that customer is:

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}$$

Decoding it:

- The numerator $b(i) - a(i)$ asks: *is the nearest other cluster farther away than my own clustermates?* If yes (the customer is well-placed), this is positive.
- Dividing by $\max\{a(i), b(i)\}$ — the larger of the two distances — squeezes the result into the range from $-1$ to $+1$.
- A silhouette near **$+1$** means the customer is deep inside its own cluster and far from any other (excellent). Near **$0$** means it sits right on the border between two clusters (ambiguous). **Negative** means it is actually closer to a different cluster than its own (probably misassigned).

The **average silhouette score** over all customers is one number summarising the whole clustering's quality, and `silhouette_score` from scikit-learn computes it for us. Higher is better. Let us compute it for each `k`.


In [ ]:
# We will compute the average silhouette score for each k from 2 to 8.
sil_scores = []

# Fit k-means for each k and measure the silhouette of the resulting clustering.
for k in k_values:
    # Same settings as the elbow plot: k clusters, fixed seed, 10 restarts.
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    # Fit and get a cluster label for every customer in one call.
    labels = km.fit_predict(X_scaled)
    # Average, over all customers, how well each sits in its cluster versus the nearest other cluster.
    sil_scores.append(silhouette_score(X_scaled, labels))

# Start a new figure for the silhouette plot.
plt.figure(figsize=(8, 5))

# Plot the silhouette score against k.
plt.plot(list(k_values), sil_scores, marker="o", color="darkorange")

# Label the axes and add a title.
plt.xlabel("number of clusters k")
plt.ylabel("average silhouette score")
plt.title("Silhouette plot: how well-separated the clusters are (higher is better)")

# Render the plot.
plt.show()

# Print the numbers.
for k, val in zip(k_values, sil_scores):
    print(f"k = {k}: silhouette = {val:.4f}")

The silhouette is highest at **`k = 2`** (about **0.29**), with **`k = 3`** close behind (about **0.26**), and then it drops more steeply for `k = 4` and beyond. So the two methods point in slightly different directions, and this is the **trade-off the brief asked us to weigh**:

- **Silhouette narrowly prefers `k = 2`.** Two clusters are the cleanest, most separated split — essentially "food-service customers" versus "everyone else."
- **The elbow prefers `k = 3`**, and so does the *business*. Splitting into three reveals a distinction that two clusters hide — as we will see, the "everyone else" group is really two different kinds of customer, and telling them apart is commercially useful.

We choose **`k = 3`**. The cost is small (silhouette slips only from about 0.29 to 0.26), and the reward is a third segment that is genuinely actionable rather than a statistical artefact. This is the honest reasoning a good analyst shows their stakeholders: *the cleanest split is two, but three buys us a real, usable distinction at a price worth paying.* Neither plot "decides" for us — they inform a judgement we then own and explain.


***
## 6.12 Fitting the final model

Decision made: **three clusters.** We now fit the final k-means model on the scaled data, attach each customer's cluster label back onto the original table (so we can profile segments in real money next), and report the basic facts about the solution.


In [ ]:
# We choose k = 3, for the reasons argued in §6.10–§6.11.
k_final = 3

# Build the final k-means model: 3 clusters, fixed seed for reproducibility, 10 restarts.
kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)

# Fit on the scaled data and get a cluster label (0, 1, or 2) for every customer.
cluster_labels = kmeans.fit_predict(X_scaled)

# Attach the labels to the ORIGINAL dataframe, so we can profile clusters in real monetary units.
wholesale["cluster"] = cluster_labels

# Count how many customers landed in each cluster.
sizes = wholesale["cluster"].value_counts().sort_index()
print("Customers per cluster:")
print(sizes)

# Show each cluster's share of the 440-customer base.
print("\nShare of customers (%):")
print((sizes / len(wholesale) * 100).round(1))

# Report the overall quality of this solution.
print(f"\nSilhouette score: {silhouette_score(X_scaled, cluster_labels):.4f}")
print(f"Inertia (within-cluster sum of squares): {kmeans.inertia_:,.1f}")

We have three segments of sensible, balanced sizes: roughly **80**, **147**, and **213** customers — about **18%, 33%, and 48%** of the base. Compare that to the degenerate **45 / 393 / 2** split we got on the raw data in §6.7: scaling turned a useless result into three groups large enough to build a strategy around. The silhouette of about **0.26** matches what the §6.11 plot promised for `k = 3`.

Cluster labels (`0`, `1`, `2`) are just arbitrary name-tags the algorithm hands out — they carry no meaning yet. The next two sections give them meaning: first by profiling what each cluster *buys* (§6.13–§6.14), then by giving each one a human name (§6.15).


***
## 6.13 Profiling the segments — what each cluster actually buys

We clustered on the *scaled* data, but scaled numbers ("1.2 standard deviations above average") are no way to brief a sales team. To understand and name the segments we bring the spending back to **real monetary units**: for each cluster, the **average spend in each category.** We add the overall average as a bottom row, so each segment can be read against the typical customer.


In [ ]:
# Group the ORIGINAL (un-scaled) spending by cluster and take the mean — interpretable money numbers.
profile = wholesale.groupby("cluster")[spend_cols].mean().round(0).astype(int)

# Add a final row with the overall average across all 440 customers, for comparison.
profile.loc["ALL"] = wholesale[spend_cols].mean().round(0).astype(int)

# Show the profile table.
profile

This table is the heart of the segmentation. Read each row against the `ALL` row at the bottom:

- **Cluster 0** (about 80 customers) spends heavily on **`Grocery`** (about 12,570 vs the 7,951 average), **`Detergents_Paper`** (about 5,554 vs 2,881), and **`Milk`** — but very little on **`Fresh`** (about 2,899 vs 12,000) or **`Frozen`** (about 607 vs 3,072). This is a **packaged-goods, shelf-stable** basket.
- **Cluster 1** (about 147 customers) sits **above average on every single category** — the biggest baskets across the board, and especially strong on `Milk`, `Grocery`, and `Delicassen`. These are the **high-volume, buy-everything** accounts.
- **Cluster 2** (about 213 customers — the largest group) is dominated by **`Fresh`** (about 11,939) and buys some **`Frozen`**, but is far below average on `Milk`, `Grocery`, and especially `Detergents_Paper` (about 424 vs 2,881). This is a **fresh-and-frozen** basket — the shape you would expect from a kitchen.

The averages already sketch three recognisable customer types. A picture will make the contrast even sharper, which is §6.14.


***
## 6.14 Visualizing the segments — the fingerprint heatmap

The profile table is in money, which is great for absolute size but makes it harder to see *what is distinctive* about each segment at a glance. For that, the **scaled** numbers are perfect: they say, for each segment and category, how far **above (+)** or **below (−)** the overall average that segment sits, in standard-deviation units. Displayed as a colour grid — a **heatmap** — they give each segment a visual **fingerprint**.


In [ ]:
# Take the scaled table and attach the cluster labels (a copy, so we do not disturb X_scaled).
scaled_with_label = X_scaled.copy()

# Add the cluster label to each row.
scaled_with_label["cluster"] = cluster_labels

# Average each scaled category within each cluster: the "fingerprint" of standardized means.
scaled_means = scaled_with_label.groupby("cluster")[spend_cols].mean()

# Start a figure sized for a wide heatmap.
plt.figure(figsize=(9, 4))

# Draw the heatmap: red = above-average spend, blue = below-average, white = about average.
# annot=True writes the number in each cell; center=0 puts the colour midpoint at the average.
sns.heatmap(scaled_means, annot=True, fmt="+.2f", cmap="RdBu_r", center=0,
            cbar_kws={"label": "standard deviations from overall average"})

# Title and axis label.
plt.title("Segment fingerprints: each cluster's spending vs the overall average")
plt.ylabel("cluster")

# Tidy spacing and render.
plt.tight_layout()
plt.show()

The fingerprints are unmistakable:

- **Cluster 0** glows red on **`Detergents_Paper`** (+0.78), **`Grocery`** (+0.68), and **`Milk`** (+0.48), and deep blue on **`Fresh`** (−1.18) and **`Frozen`** (−1.14). A grocery-and-household-goods profile.
- **Cluster 1** is red almost everywhere — above average on all six categories, strongest on **`Milk`** (+0.80), **`Grocery`** (+0.74), and **`Delicassen`** (+0.68). The buy-everything profile.
- **Cluster 2** is blue across `Milk`, `Grocery`, `Detergents_Paper` and `Delicassen`, with `Fresh` and `Frozen` near or just above the line. A lean, fresh-focused profile.

Three categories — `Fresh` and `Frozen` versus `Detergents_Paper` and `Grocery` — do most of the work of telling these segments apart. We now have enough to give the segments names and to put them to the test.


***
## 6.15 Naming the segments and an external-validation surprise

From the profile (§6.13) and the fingerprints (§6.14), three names write themselves:

- **Cluster 0 → "Grocery & Household-Goods Retailers."** High on grocery, detergents-and-paper, and milk; almost no fresh or frozen. The basket of a shop selling packaged goods off shelves.
- **Cluster 1 → "High-Volume All-Rounders."** Above average on everything — the large accounts that buy across the whole catalogue.
- **Cluster 2 → "Fresh-Focused Food Service."** Fresh and frozen dominate; grocery and detergents are minimal. The basket of a kitchen — a restaurant, café, or hotel.

Those names came **purely from spending patterns.** Now for the test we set up back in §6.3, when we deliberately hid the `Channel` column (`1` = Horeca / Hotel-Restaurant-Café, `2` = Retail) from the algorithm. K-means **never saw** which customers were food-service and which were shops. So here is the question: *did our spending-only segmentation rediscover that business distinction on its own?* Let us cross-tabulate cluster against the hidden `Channel` (and `Region`, while we are at it).


In [ ]:
# Cross-tabulate cluster against the Channel column we hid from the model (1 = Horeca, 2 = Retail).
channel_tab = pd.crosstab(wholesale["cluster"], wholesale["Channel"])

# Rename the columns to their meanings so the table reads clearly.
channel_tab.columns = ["Horeca (1)", "Retail (2)"]

# Show the cluster-by-Channel table.
print("Cluster vs Channel (Channel was NOT used in clustering):")
print(channel_tab)

# Cross-tabulate cluster against Region (1 = Lisbon, 2 = Oporto, 3 = Other).
region_tab = pd.crosstab(wholesale["cluster"], wholesale["Region"])

# Rename the Region columns too.
region_tab.columns = ["Lisbon (1)", "Oporto (2)", "Other (3)"]

# Show the cluster-by-Region table.
print("\nCluster vs Region (Region was NOT used in clustering):")
print(region_tab)

This is the satisfying payoff. Look at **Cluster 2**, our "Fresh-Focused Food Service" segment: it contains **210 Horeca customers and only 3 Retail** — about **98.6% Horeca**. The algorithm, working from spending alone, almost perfectly isolated the hotels, restaurants, and cafés. And from the other side: of the **142 Retail customers**, **139** (about **98%**) fall into Clusters 0 and 1, the two retail-flavoured segments. **K-means rediscovered the Horeca-vs-Retail business distinction without ever being told it existed.** That is strong evidence the segmentation reflects something real about how these businesses operate, not an arbitrary slicing.

The **Region** table tells the opposite — and equally useful — story. Each cluster spreads across Lisbon, Oporto, and Other in roughly the same proportions as the whole base. Geography does **not** separate these customers; *what they buy* does. That is worth telling the commercial team plainly: segment by purchasing behaviour, not by postcode.

Let us attach the human-readable names and measure how the **revenue** is distributed across the three segments — the number the Head of Commercial will care about most.


In [ ]:
# Map each cluster number to the name we read off the profile and fingerprints.
segment_names = {
    0: "Grocery & Household-Goods Retailers",
    1: "High-Volume All-Rounders",
    2: "Fresh-Focused Food Service",
}

# Add a readable segment-name column to the dataframe.
wholesale["segment"] = wholesale["cluster"].map(segment_names)

# Total annual spend captured by each segment (sum across all six categories, summed over its customers).
segment_revenue = wholesale.groupby("segment")[spend_cols].sum().sum(axis=1).sort_values(ascending=False)

# Show the total annual spend per segment.
print("Total annual spend per segment (m.u.):")
print(segment_revenue.astype(int))

# Show each segment's share of total spend.
print("\nShare of total annual spend (%):")
print((segment_revenue / segment_revenue.sum() * 100).round(1))

The revenue split reframes the whole picture. The **High-Volume All-Rounders** are only about **33%** of customers (147 of 440) but account for roughly **53%** of all spend — over half the business sits in one segment. The **Fresh-Focused Food Service** group, though the *largest* by headcount (about 48% of customers), contributes about **31%** of spend, and the **Grocery & Household-Goods Retailers** about **16%**. *Number of customers and share of revenue are not the same thing* — a distinction that should drive where the sales team spends its energy, and exactly the kind of insight we will hand the model to turn into a playbook next.


***
## 6.16 Our API calls — Anthropic turns the profiles into a sales playbook

> **Callback to §1.11, §2.13, §3.12, §4.16, §5.20.** Session 01 made plain-text Anthropic calls. Session 02 asked for a JSON outline and validated it in Python. Session 03 stepped up to **schema-enforced** structured output using **Anthropic's tool use** with a Pydantic schema (§3.12) — the model literally cannot return something off-schema. Sessions 04 and 05 reused that exact pattern for the drivers memo and the churn policy. **Same pattern this week**, applied to a per-segment sales playbook.

> **Why Anthropic only.** This course uses **Anthropic Claude exclusively** — no second provider, no second SDK, no second key. The Messages API plus tool use gives us everything: plain-text generation for language, and schema-enforced output for structured deliverables. (This is course **Principle 20** in the instructor's notes.)

We make **two API calls**, both to Anthropic:

1. **Call 1 — schema-enforced segmentation playbook.** Anthropic turns the segment profiles from §§6.13–6.15 into a structured playbook — for each segment an evidence-based **name**, a **who-they-are** description, a **sales action**, and a **retention action** — as a JSON object that *conforms to a Pydantic schema we define in Python.*
2. **Call 2 — plain-text headline paragraph.** Anthropic drafts the *"Headline"* paragraph of the commercial memo from the numbers we computed.

The unbreakable rule is the same as every prior session: **"Python computes, the model interprets."** We pass the model the numbers we calculated; it names and strategises, but it never invents a figure. The Segmentation Strategist persona we give it is told, in so many words, to name each segment **only** from the spending evidence supplied.

### Step 1 — verify the API key is available


In [ ]:
# Import os so we can read environment variables.
import os

# Read the Anthropic API key from the environment.
anth_key = os.environ.get("ANTHROPIC_API_KEY")

# If the key is missing, halt the notebook with a clear, actionable message (do not silently continue).
if not anth_key:
    raise SystemExit(
        "ANTHROPIC_API_KEY is not set in your environment.\n"
        "Follow Step 4 of the repository README, close VS Code, reopen, and re-run this cell."
    )

# Confirm the key is set without printing its value.
print(f"ANTHROPIC_API_KEY is set. Key length: {len(anth_key)} characters.")

### Step 2 — create the Anthropic client


In [ ]:
# Import the official Anthropic Python SDK.
import anthropic

# Create the Anthropic client; it picks up ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

# Print a confirmation that the client object was created successfully.
print("Anthropic client ready.")

### Step 3 — define a Pydantic schema for the playbook

> **Callback to §3.12.** Pydantic gives us the JSON Schema for free: we write ordinary Python classes, and `.model_json_schema()` returns the schema dict Anthropic's tool-use feature expects. New this week, we use a **nested** schema — a playbook that *contains a list of segments* — because our deliverable naturally has one entry per segment. A class can hold a list of another class; Pydantic and Anthropic handle the nesting for us.

- **`SegmentPlay`** describes **one** segment: its `segment_name`, a `who_they_are` description, a `sales_play`, and a `retention_play`.
- **`SegmentationPlaybook`** is the whole deliverable: a `segments` list (one `SegmentPlay` each), a `portfolio_note` about where revenue concentrates, and `caveats`.


In [ ]:
# Import Pydantic's BaseModel and Field helpers (installed in the `ailab` venv as an Anthropic SDK dependency).
from pydantic import BaseModel, Field

# typing.List lets us declare "a list of SegmentPlay objects" in the schema.
from typing import List

# Describe ONE segment of the playbook.
class SegmentPlay(BaseModel):
    segment_name: str = Field(
        description="A short, evidence-based name for this segment, justified ONLY by its spending profile — "
                    "name the two or three product categories that dominate or are notably absent from its basket. "
                    "Do not invent attributes that are not in the numbers."
    )
    who_they_are: str = Field(
        description="One to three sentences describing this segment in plain business language, grounded in the "
                    "per-category spend numbers provided. State which categories they spend most and least on."
    )
    sales_play: str = Field(
        description="One concrete sales action tailored to this segment's basket, two to three sentences, "
                    "referencing the categories they actually buy."
    )
    retention_play: str = Field(
        description="One concrete retention action for this segment, two to three sentences, sized to how large "
                    "and how valuable the segment is."
    )

# Describe the WHOLE playbook: a list of segments plus a portfolio note and caveats.
class SegmentationPlaybook(BaseModel):
    segments: List[SegmentPlay] = Field(
        description="Exactly three entries, one per segment, in the same order the segments are presented in the data."
    )
    portfolio_note: str = Field(
        description="Two to three sentences on where annual spend concentrates across the three segments and what "
                    "that implies for where to focus commercial effort. Use only the totals provided."
    )
    caveats: str = Field(
        description="Two to three honesty caveats: that this DESCRIBES these 440 customers and is not a forecast "
                    "about new customers; that k-means draws hard boundaries through what is really a continuum of "
                    "behaviour; and that the segments reflect spending patterns only, not profitability or strategic value."
    )

# Print confirmation that the schema is defined.
print("Pydantic schema SegmentationPlaybook (with nested SegmentPlay) defined.")

### Step 4 — assemble the numbers and call Anthropic with a forced tool

The mechanics match §3.12, §4.16, and §5.20 exactly. First a **safety re-fit** cell rebuilds every number this section needs (so it runs correctly even after a kernel restart), and assembles the per-segment figures into a plain-text block — *the only numbers the model is allowed to use.* Then we build a tool from our Pydantic schema and force the model to call it (`tool_choice`), so the reply must be a schema-valid playbook.


In [ ]:
# Safety re-fit — rebuild every number this section needs, regardless of run order.
# (All computed earlier: pipeline §6.7, fit §6.12, profile §6.13, validation §6.15. Re-deriving is cheap.)

# The six spending columns.
spend_cols = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Log-transform then standardize (the §6.7 pipeline).
wholesale_log = np.log1p(wholesale[spend_cols])
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(wholesale_log), columns=spend_cols)

# Re-fit the final k = 3 model and attach labels.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
wholesale["cluster"] = kmeans.fit_predict(X_scaled)

# The segment names from §6.15.
segment_names = {0: "Grocery & Household-Goods Retailers",
                 1: "High-Volume All-Rounders",
                 2: "Fresh-Focused Food Service"}

# Per-cluster pieces: mean spend, sizes, total revenue, and the Channel mix.
profile = wholesale.groupby("cluster")[spend_cols].mean().round(0).astype(int)
sizes = wholesale["cluster"].value_counts().sort_index()
revenue = wholesale.groupby("cluster")[spend_cols].sum().sum(axis=1)
total_rev = revenue.sum()
overall = wholesale[spend_cols].mean().round(0).astype(int)
channel_tab = pd.crosstab(wholesale["cluster"], wholesale["Channel"])

# Build the numbers summary the model is allowed to see (Python computes, the model interprets).
lines = []
lines.append(
    f"K-means segmentation of n = {len(wholesale)} wholesale customers into 3 segments, using annual spend "
    f"across six product categories (log-transformed, then standardized). This DESCRIBES these {len(wholesale)} "
    f"customers; it is not a forecast about new customers."
)
lines.append("Overall average spend (m.u.): " + ", ".join(f"{c} {overall[c]:,}" for c in spend_cols))
for cid in [0, 1, 2]:
    nm = segment_names[cid]
    n = int(sizes[cid])
    rev = int(revenue[cid])
    means = ", ".join(f"{c} {profile.loc[cid, c]:,}" for c in spend_cols)
    hor = int(channel_tab.loc[cid, 1])
    ret = int(channel_tab.loc[cid, 2])
    lines.append(
        f"\nSegment {cid} '{nm}': {n} customers ({n/len(wholesale)*100:.1f}% of base); "
        f"total annual spend {rev:,} m.u. ({rev/total_rev*100:.1f}% of all spend). "
        f"Average spend per customer: {means}. Channel mix: {hor} Horeca, {ret} Retail."
    )

# Join into one block of text and show it.
segment_numbers = "\n".join(lines)
print(segment_numbers)

In [ ]:
# Build the Anthropic tool spec from our Pydantic schema.
# model_json_schema() returns a JSON Schema dict — exactly what Anthropic's `input_schema` expects.
playbook_tool = {
    "name": "submit_segmentation_playbook",
    "description": "Submit segment profiles and a short sales playbook. You MUST call this tool with every required field filled in.",
    "input_schema": SegmentationPlaybook.model_json_schema(),
}

# Call Claude. `tool_choice` forces the model to call this specific tool — it cannot reply with free text.
playbook_response = client.messages.create(
    model="claude-haiku-4-5",                                            # same Haiku model as §§1.11–5.20
    max_tokens=2000,                                                     # room for three segments plus notes
    system=(                                                             # the Segmentation Strategist persona
        "You are a Segmentation Strategist Assistant for a wholesale food-and-goods distributor. "
        "You turn a computed customer segmentation into evidence-based segment names and a short, practical "
        "sales playbook. You enforce four disciplines: "
        "(1) name and describe each segment using ONLY the spending numbers given to you — never invent traits; "
        "(2) make every sales and retention action specific to the categories that segment actually buys; "
        "(3) frame everything as a description of THESE customers, never a forecast about new or future customers; "
        "(4) never invent numbers — use only those provided."
    ),
    tools=[playbook_tool],                                               # the only tool the model may call
    tool_choice={"type": "tool", "name": "submit_segmentation_playbook"},# force this specific tool
    messages=[
        {
            "role": "user",
            "content": (
                "Here is a k-means segmentation of a wholesale customer base:\n\n"
                + segment_numbers
                + "\n\nFill in the segmentation playbook. Provide exactly three segment entries in the order shown, "
                + "and use only the numbers above."
            ),
        }
    ],
)

# Scan the response for the tool_use block. Because tool_choice forced our tool, exactly one exists.
playbook_dict = None
for block in playbook_response.content:
    if block.type == "tool_use":
        playbook_dict = block.input    # this dict already conforms to our schema — Anthropic enforced it
        break

# Wrap the dict back into our Pydantic class for type-safe access.
playbook = SegmentationPlaybook(**playbook_dict)

# Print the playbook segment by segment.
for i, seg in enumerate(playbook.segments):
    print(f"== Segment {i}: {seg.segment_name} ==")
    print(f"   Who they are: {seg.who_they_are}")
    print(f"   Sales play:   {seg.sales_play}")
    print(f"   Retention:    {seg.retention_play}\n")

# Print the portfolio note and the caveats.
print("Portfolio note:", playbook.portfolio_note)
print("\nCaveats:", playbook.caveats)

**What we see.** Anthropic returns a tool call whose `input` is a JSON object matching our nested schema exactly: a list of three segment entries plus the portfolio note and caveats. Because `tool_choice` forced the tool, no validation step is needed — the model *cannot* return something off-schema. Same generation-time guarantee as §3.12, §4.16, and §5.20, now with a nested list.

### Call 2 — Anthropic drafts the memo headline paragraph

Now we hand the same computed numbers to Anthropic and ask for a short plain-text paragraph for the *"Headline"* of the commercial memo. Same *"Python computes, the model interprets"* rule.


In [ ]:
# Call Anthropic to write the headline paragraph for the commercial memo.
memo_response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    system=(
        "You translate a customer segmentation into one short paragraph for a commercial director's memo. "
        "Lead with how many segments there are and which one drives the most revenue. Name each segment by what it buys. "
        "Close with a one-sentence honesty caveat framing this as a description of these customers, not a forecast. "
        "Use plain English and plain numbers. Use ONLY the numbers given to you; never invent new numbers."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                "Here is the segmentation:\n\n"
                + segment_numbers
                + "\n\nWrite ONE short paragraph (4 to 6 sentences) for the 'Headline' section of the commercial "
                + "director's memo. Lead with the number of segments and which one drives the most revenue, name "
                + "each segment by what it buys, and close with an honest caveat."
            ),
        }
    ],
)

# Print Anthropic's paragraph.
print(memo_response.content[0].text)

**What we see.** A short, plain-English paragraph that leads with the three segments and the revenue concentration, names each by its basket, and closes with the honest *"this describes these customers"* caveat. The model invented **no numbers** — it used only the set we computed and handed it.

> **Mini-recap of §6.16.** Two API calls, one provider. Anthropic tool use with a (nested) Pydantic-derived schema **guarantees** the structured playbook at generation time; plain-text Anthropic drafts the headline paragraph from computed numbers. Same *"Python computes, the model interprets"* discipline as Sessions 01–05; same single-provider architecture as §3.12, §4.16, and §5.20. (Principle 20.)


***
## 6.17 The segmentation story and sales playbook — fully worked

It is Friday afternoon. Here are the two artifacts you would hand the Head of Commercial. Both are grounded entirely in numbers computed in this notebook.

---

### Artifact 1 — Customer segmentation playbook (three segments)

We grouped all **440 clients** into three segments using their annual spend across six product categories. K-means never saw whether a client was a restaurant or a shop, yet it recovered that split almost perfectly (Segment 3 below is **98.6% Horeca**).

**Segment 1 — "Fresh-Focused Food Service"** · 213 clients (48% of base) · ~31% of revenue
*Who they are.* Kitchens — restaurants, cafés, hotels (210 of 213 are Horeca). They spend heavily on **`Fresh`** (avg ~11,939 m.u.) and buy some **`Frozen`**, but almost nothing on grocery, milk, or detergents (avg `Detergents_Paper` ~424 vs the 2,881 base average).
*Sales play.* Lead with **fresh produce and frozen range** — quality, reliability of supply, and delivery frequency. Do not waste calls pitching packaged grocery or cleaning lines; they are not in this basket.
*Retention play.* This is the largest group by headcount, so protect it with **service reliability** — guaranteed morning delivery windows and substitution policies for fresh items, where a single missed delivery can lose a restaurant.

**Segment 2 — "High-Volume All-Rounders"** · 147 clients (33% of base) · ~53% of revenue
*Who they are.* The largest accounts, **above average on every category** (especially `Milk`, `Grocery`, `Delicassen`). A third of the customers, but **over half of all spend** sits here.
*Sales play.* These accounts already buy broadly; grow them with **bundled catalogue deals and volume tiers** that reward consolidating more of their order with you across categories.
*Retention play.* Highest-stakes segment — disproportionate revenue per client. Assign **named account managers** and review these relationships first; losing one all-rounder costs far more than losing one food-service client.

**Segment 3 — "Grocery & Household-Goods Retailers"** · 80 clients (18% of base) · ~16% of revenue
*Who they are.* Shops selling packaged goods: high on **`Grocery`** (avg ~12,570), **`Detergents_Paper`** (~5,554), and `Milk`, but very low on `Fresh` (~2,899) and `Frozen` (~607).
*Sales play.* Pitch the **shelf-stable and household range** — grocery staples, detergents, paper goods — with promotions on case quantities. Fresh and frozen are not their business.
*Retention play.* The smallest group; serve efficiently with **standardised reorder schedules and self-service ordering** rather than high-touch account management.

**Portfolio note.** Effort should follow revenue, not headcount. The **High-Volume All-Rounders** are a third of clients but over half of spend — they deserve the most senior attention. The **Fresh-Focused** group is largest by count but mid-sized by revenue; serve it with reliable logistics rather than heavy sales time.

**Caveats.** This segmentation **describes these 440 clients** as they spent this year; it is not a forecast about new clients. K-means draws **hard lines through what is really a continuum** — clients near a boundary could reasonably belong to either side. And the segments reflect **spending patterns only** — not profit margin or strategic value, which would refine the picture further.

---

### Artifact 2 — Commercial memo

**To:** Head of Commercial
**From:** [Your name], Junior Analyst
**Re:** Customer segmentation — three types, and what to do about each

**Headline.** Our 440 clients fall into three clear segments based on what they buy. **High-Volume All-Rounders** (147 clients, a third of the base) generate **over half of all spend** and should get our most senior attention. **Fresh-Focused Food Service** clients (213 — nearly half the base, and 99% of them hotels/restaurants/cafés) are our largest group by count and about a third of revenue; they live on fresh and frozen, so we win them with supply reliability, not catalogue breadth. **Grocery & Household-Goods Retailers** (80 clients, ~16% of revenue) buy packaged staples and cleaning lines and can be served efficiently with standing orders. Notably, the model was never told which clients were restaurants versus shops, yet it separated them almost perfectly from spending alone — strong evidence these segments are real. One honest caveat: this describes the clients we have today, not a prediction about new ones, and the boundaries between segments are softer than three tidy labels suggest.

---

Both artifacts use only numbers this notebook computed. That is the whole discipline of the course in one deliverable: **Python found the segments and the figures; the model helped name and narrate them; every claim traces back to a cell you can re-run.**


***
## 6.18 References — what to study to deepen this session

A short, focused list. If you watch only two videos, make them the StatQuest *Gentle Introduction to Machine Learning* (for the supervised-vs-unsupervised framing of §6.5) and *K-means clustering* (for the assign-and-update loop of §6.8).

### StatQuest videos (YouTube)

| Video | What it clarifies |
|---|---|
| [A Gentle Introduction to Machine Learning](https://www.youtube.com/watch?v=Gv9_4yMHFhI) | The big map of machine learning, including where unsupervised methods like clustering sit. Pairs with §6.5. |
| [K-means clustering](https://www.youtube.com/watch?v=4b5d3muPQmA) | The assign-and-update loop, how centroids move, and how to pick k — exactly §§6.8–6.11, in pictures. |

### Khan Academy

- [Clusters in scatter plots](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/introduction-to-scatterplots/a/scatterplots-and-correlation-review) — a gentle, visual grounding for what a "cluster" looks like in data, reinforcing the intuition behind §6.6.
- [Statistics & Probability hub](https://www.khanacademy.org/math/statistics-probability) — the full track; the data-visualization and interpretation units are what make a segmentation credible to stakeholders.

### Documentation

- [scikit-learn `KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) — the algorithm we fit in §§6.10–6.12, including the `n_init` and `random_state` arguments.
- [scikit-learn `StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) — the z-score machine behind §6.7.
- [scikit-learn `silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html) — the cluster-quality measure of §6.11.
- [scikit-learn: Clustering](https://scikit-learn.org/stable/modules/clustering.html) — the wider guide, with a clear discussion of k-means' assumptions and limitations.
- [Anthropic tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) — the schema-enforced output mechanism behind §6.16's playbook.

### Dataset

- [Wholesale customers dataset, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/292/wholesale+customers) — 440 clients of a Portuguese wholesale distributor, annual spend across six product categories, licensed **CC BY 4.0**. The `Channel` and `Region` columns are the ones we deliberately held back for the §6.15 validation.

> See you in **Session 07**, where we **keep these exact segments** but learn to *show* them. We will use **t-SNE** to squeeze the six spending dimensions down to a 2D picture you can put in front of non-technical stakeholders — a map where each customer is a dot and the segments you found this week appear as visible islands. The segmentation work is done; next week is about making it land in a meeting.


<hr>

![](../_img/DK_Logo_White_150.png)

DataKolektiv, 2026.

[hello@datakolektiv.com](mailto:hello@datakolektiv.com)


<font size=1>License: [GPLv3](../LICENSE). This Notebook is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version. This Notebook is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.</font>
